In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/BigContest_JSH/BigContest/data/transit')

import dask.dataframe as dd
import time
import pandas as pd

import matplotlib.pyplot as plt
import numpy as np

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install koreanize-matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 48.0 MB/s eta 0:00:00


In [3]:
df=pd.read_csv('transit.csv')
df

FileNotFoundError: [Errno 2] No such file or directory: 'transit.csv'

In [ ]:
cd_origin = df['routeID'].str[:-2]
cd_origin = df.copy()
cd_origin['routeID'] = cd_origin['routeID'].str[:-2]
cd_origin

In [ ]:
# Group by the modified 'routeID' and calculate total time per routeID
grouped_cd_origin = cd_origin.groupby('routeID').agg(totalDistance=('time', 'sum')).reset_index()

# Calculate the weighted median of 'time' for each modified routeID
cd_origin['time'] = cd_origin['time'].fillna(0)
cd_origin_weighted_median = cd_origin.groupby('routeID').apply(
    lambda x: weighted_median(x['time'], x['time'])
).reset_index(name='weighted_median')

# Display the results
cd_origin_weighted_median


In [ ]:
import pandas as pd
# 1. 데이터의 'time' 값을 5개 구간으로 나누기
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']
cd_origin['dist_group'] = pd.cut(cd_origin['time'], bins=bins, labels=group_names)

# 2. 각 그룹별 임의의 가중치를 설정
# 여기서는 그룹별 가중치를 임의로 설정 (예시: group_A=1, group_B=2, group_C=3, ...)
group_weights = {
    'group_A': 1,
    'group_B': 2,
    'group_C': 3,
    'group_D': 4,
    'group_E': 5
}

# 3. dist_group에 맞는 가중치를 할당
cd_origin['weight'] = cd_origin['dist_group'].map(group_weights)

# 4. 가중 중간값을 계산하는 함수 (가중치를 고려한 중간값)
def weighted_median(values, weights):
    sorted_indices = np.argsort(values)
    sorted_values = np.array(values)[sorted_indices]
    sorted_weights = np.array(weights)[sorted_indices]

    cumulative_weight = np.cumsum(sorted_weights)
    total_weight = cumulative_weight[-1]

    # 가중치의 중간값을 찾기 위해 cumulative_weight가 total_weight의 절반을 초과하는 지점 찾기
    median_idx = np.searchsorted(cumulative_weight, total_weight / 2)

    return sorted_values[median_idx]

# 5. routeID별로 그룹화하여 가중 중간값 계산
def calculate_weighted_median_by_route(df):
    result = []
    for routeID in df['routeID'].unique():
        # 해당 routeID에 해당하는 time 값과 가중치
        route_data = df[df['routeID'] == routeID]

        # 가중 중간값 계산
        weighted_med = weighted_median(route_data['time'], route_data['weight'])
        result.append({'routeID': routeID, 'weighted_median': weighted_med})

    return pd.DataFrame(result)

# 결과 출력
cd_origin_weighted_median = calculate_weighted_median_by_route(cd_origin)
cd_origin_weighted_median

In [ ]:
# 1. routeID별로 임의의 가중치를 설정 (각 routeID에 대해 고유한 가중치 설정)
# 예시로 각 routeID별로 임의로 가중치를 설정
route_weights = {
    'BB': [1, 2, 3],   # routeID 'BB'에 대한 가중치
    'BD': [1, 2, 3],
    'BGA': [1, 1.5, 2],
    'BGB': [1.2, 2, 2.5],
    'BI': [1, 1.5, 2],
    'BS': [1.1, 1.8, 2.2],
    'DD': [1, 1.2, 1.5],
    'GGA': [2, 2.5, 3],
    'GGB': [1.5, 2, 2.5],
    'II': [2, 2.5, 3],
    'SB': [1.5, 2.2, 2.8],
    'SD': [1.2, 1.5, 1.8],
    'SGA': [1, 1.2, 1.5],
    'SGB': [1.3, 1.6, 2],
    'SI': [1.5, 1.8, 2],
    'SS': [1, 1.2, 1.4]
}

# 2. 각 routeID에 대해 가중치를 할당 (가중치를 순차적으로 할당)
cd_origin['weight'] = cd_origin.apply(lambda row: route_weights[row['routeID']][0], axis=1)

# 3. 가중 중간값을 계산하는 함수 (가중치를 고려한 중간값)
def weighted_median(values, weights):
    sorted_indices = np.argsort(values)
    sorted_values = np.array(values)[sorted_indices]
    sorted_weights = np.array(weights)[sorted_indices]

    cumulative_weight = np.cumsum(sorted_weights)
    total_weight = cumulative_weight[-1]

    # 가중치의 중간값을 찾기 위해 cumulative_weight가 total_weight의 절반을 초과하는 지점 찾기
    median_idx = np.searchsorted(cumulative_weight, total_weight / 2)

    return sorted_values[median_idx]

# 4. routeID별로 그룹화하여 가중 중간값 계산
cd_origin_weighted_median = cd_origin.groupby('routeID').apply(
    lambda x: weighted_median(x['time'], x['weight'])
).reset_index(name='weighted_median')

# 결과 출력
cd_origin_weighted_median

In [ ]:
rt=pd.read_csv('tot_car.csv')
rt

In [ ]:
# 지수 표기 형식으로 된 'origin_hdong_cd' 값을 정수형으로 변환 (+) string 변환
rt['origin_hdong_cd'] = rt['origin_hdong_cd'].astype(int).astype('string')
rt['dest_hdong_cd'] = rt['dest_hdong_cd'].astype(int).astype('string')

In [ ]:
rt = rt.reset_index(drop=True).drop(columns = 'Unnamed: 0')
rt

In [ ]:
dest_hdong_map = {
    '1156054000': 'S',
    '4575034000': 'I',
    '5115057200': 'GA',
    '5115058000': 'GB',
    '2635052000': 'B',
    '3020055000': 'D'
}

In [ ]:
# origin_hdong_map 정의
origin_hdong_map = {
    51: 'G',
    30: 'D',
    45: 'I',
    26: 'B',
    11: 'S'
}

In [ ]:
rt['목적지'] = rt['dest_hdong_cd'].map(dest_hdong_map)
# rt['출발지'] = rt['origin_hdong_cd'].map(origin_hdong_map2)
rt

In [ ]:
rt['출발지'] = rt['origin_cd'].map(origin_hdong_map)
rt

In [ ]:
rt= rt[rt['modal'] != 0]
rt

In [ ]:
dff_b = rt[(rt['출발지'] == 'B') & (rt['목적지']=='B')]
dff_b

In [ ]:
df_bd=rt[(rt['출발지']=='B') & (rt['목적지']== 'D')]
df_bd

In [ ]:
df_bga = rt[(rt['출발지'] == 'B') & (rt['목적지'] == 'GA')]
df_bga

In [ ]:
df_bgb = rt[(rt['출발지'] == 'B') & (rt['목적지'] == 'GB')]
df_bgb

In [ ]:
df_bga = rt[(rt['출발지'] == 'B') & (rt['목적지'] == 'GA')]
df_bga

In [ ]:
df_bI = rt[(rt['출발지'] == 'B') & (rt['목적지'] == 'I')]
df_bI

In [ ]:
df_bs = rt[(rt['출발지'] == 'B') & (rt['목적지'] == 'S')]
df_bs

In [ ]:
df_dd=rt[(rt['출발지']=='D') & (rt['목적지']=='D')]
df_dd

In [ ]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_dd['dist_group'] = pd.cut(df_dd['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_dd['weight'] = df_dd['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_dd['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg = np.average(df_dd['od_dist_avg'], weights=df_dd['weight'])
print("가중 평균:", weighted_avg)

# 가중치를 숫자형으로 변환
df_dd['weight'] = df_dd['weight'].astype(float)

# 가중 중간값 계산
weighted_median = np.median(np.repeat(df_dd['od_dist_avg'], (df_dd['weight'] * 100).astype(int)))  # 가중치를 반영하여 중간값 계산
print("가중 중간값:", weighted_median)

In [ ]:
df_gga = rt[(rt['출발지'] == 'G') & (rt['목적지']=='GA')]
df_gga

In [ ]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_gga['dist_group'] = pd.cut(df_gga['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_gga['weight'] = df_gga['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_gga['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg = np.average(df_gga['od_dist_avg'], weights=df_gga['weight'])
print("가중 평균:", weighted_avg)

# 가중치를 숫자형으로 변환
df_gga['weight'] = df_gga['weight'].astype(float)

# 가중 중간값 계산
weighted_median = np.median(np.repeat(df_gga['od_dist_avg'], (df_gga['weight'] * 100).astype(int)))  # 가중치를 반영하여 중간값 계산
print("가중 중간값:", weighted_median)

In [ ]:
df_ggb= rt[(rt['출발지'] == 'G') & (rt['목적지']=='GB')]
df_ggb

In [ ]:
dff_I=rt[(rt['출발지']=='I') & (rt['목적지']=='I')]
dff_I

In [ ]:
max_distance = dff_I['od_dist_avg'].max()
min_distance = dff_I['od_dist_avg'].min()

print(f"최대 이동거리: {max_distance}")
print(f"최소 이동거리: {min_distance}")

In [ ]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
dff_I['dist_group'] = pd.cut(dff_I['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
dff_I['weight'] = dff_I['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
dff_I['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg = np.average(dff_I['od_dist_avg'], weights=dff_I['weight'])
print("가중 평균:", weighted_avg)

# 가중치를 숫자형으로 변환
dff_I['weight'] = dff_I['weight'].astype(float)

# 가중 중간값 계산
weighted_median = np.median(np.repeat(dff_I['od_dist_avg'], (dff_I['weight'] * 100).astype(int)))  # 가중치를 반영하여 중간값 계산
print("가중 중간값:", weighted_median)

In [ ]:
df_ss = rt[(rt['출발지'] == 'S') & (rt['목적지']=='S')]
df_ss

In [ ]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_ss['dist_group'] = pd.cut(df_ss['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_ss['weight'] = df_ss['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_ss['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg = np.average(df_ss['od_dist_avg'], weights=df_ss['weight'])
print("가중 평균:", weighted_avg)

# 가중치를 숫자형으로 변환
df_ss['weight'] = df_ss['weight'].astype(float)

# 가중 중간값 계산
weighted_median = np.median(np.repeat(df_ss['od_dist_avg'], (df_ss['weight'] * 100).astype(int)))  # 가중치를 반영하여 중간값 계산
print("가중 중간값:", weighted_median)

In [ ]:
df_sb = rt[(rt['출발지'] == 'S') & (rt['목적지']=='B')]
df_sb

In [ ]:
df_sd = rt[(rt['출발지'] == 'S') & (rt['목적지']=='D')]
df_sd

In [ ]:
df_sga = rt[(rt['출발지'] == 'S') & (rt['목적지']=='GA')]
df_sga

In [ ]:
df_sgb = rt[(rt['출발지'] == 'S') & (rt['목적지']=='GB')]
df_sgb

In [ ]:
df_sI = rt[(rt['출발지'] == 'S') & (rt['목적지']=='I')]
df_sI